In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

import joblib

In [2]:
df = pd.read_csv("../data/processed/atm_model_ready.csv")

df.head()

,ATM Name,Transaction Date,No Of Withdrawals,No Of XYZ Card Withdrawals,No Of Other Card Withdrawals,Total amount Withdrawn,Amount withdrawn XYZ Card,Amount withdrawn Other Card,Weekday,Festival Religion,...,Rolling_7_Mean,Rolling_14_Mean,Rolling_30_Mean,Lag_1_Day,Lag_7_Day,Lag_14_Day,Lag_30_Day,Rolling_7_Day_Mean,Rolling_14_Day_Mean,Rolling_30_Day_Mean
0,Airport ATM,2011-01-15,52,35,17,3494.62,2991.72,502.90,Saturday,H,...,9288.517143,9561.367143,NaN,9811.90,11104.46,10772.76,NaN,9288.517143,9561.367143,9561.367143
1,Airport ATM,2011-01-16,42,33,9,3297.74,2841.92,455.82,Sunday,H,...,8201.397143,9041.500000,NaN,3494.62,7213.94,5748.04,NaN,8201.397143,9041.500000,9156.917333
2,Airport ATM,2011-01-17,40,24,16,2140.00,1703.44,436.56,Monday,H,...,7641.940000,8866.478571,NaN,3297.74,9077.88,12923.46,NaN,7641.940000,8866.478571,8790.718750
3,Airport ATM,2011-01-18,86,65,21,7342.34,5844.34,1498.00,Tuesday,NH,...,6650.814286,8096.231429,NaN,2140.00,8360.98,11590.24,NaN,6650.814286,8096.231429,8399.500000
4,Airport ATM,2011-01-19,106,86,20,9178.46,8209.04,969.42,Wednesday,NH,...,6505.294286,7792.810000,NaN,7342.34,10640.08,11342.00,NaN,6505.294286,7792.810000,8340.768889


In [3]:
print("Shape:", df.shape)
print(df.isnull().sum())

Shape: (10658, 33)
ATM Name                          0
Transaction Date                  0
No Of Withdrawals                 0
No Of XYZ Card Withdrawals        0
No Of Other Card Withdrawals      0
Total amount Withdrawn            0
Amount withdrawn XYZ Card         0
Amount withdrawn Other Card       0
Weekday                           0
Festival Religion                 0
Working Day                       0
Holiday Sequence                  0
Year                              0
Month                             0
Day                               0
Day_of_Week                       0
Is_Weekend                        0
Week_of_Year                      0
Demand_Outlier                    0
Lag_1                             0
Lag_7                             0
Lag_14                            0
Lag_30                           80
Rolling_7_Mean                    0
Rolling_14_Mean                   0
Rolling_30_Mean                  80
Lag_1_Day                         0
Lag_7_Day

In [4]:
df["Transaction Date"] = pd.to_datetime(
    df["Transaction Date"],
    errors="coerce"
)

In [5]:
target = "Total amount Withdrawn"

In [6]:
df = df.sort_values("Transaction Date").reset_index(drop=True)

In [7]:
print("Start:", df["Transaction Date"].min())
print("End:", df["Transaction Date"].max())

Start: 2011-01-15 00:00:00
End: 2017-09-29 00:00:00


In [8]:
n = len(df)

train_end = int(n * 0.70)
validation_end = int(n * 0.85)

train_df = df.iloc[:train_end]
validation_df = df.iloc[train_end:validation_end]
test_df = df.iloc[validation_end:]

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

Train: (7460, 33)
Validation: (1599, 33)
Test: (1599, 33)


In [9]:
features = [
    "ATM Name",
    "Lag_1_Day",
    "Lag_7_Day",
    "Lag_14_Day",
    "Lag_30_Day",
    "Rolling_7_Day_Mean",
    "Rolling_14_Day_Mean",
    "Rolling_30_Day_Mean",
    "Year",
    "Month",
    "Day",
    "Day_of_Week",
    "Is_Weekend",
    "Week_of_Year",
    "Working Day",
    "Holiday Sequence",
    "Festival Religion"
]

In [10]:
X_train = train_df[features]
y_train = train_df[target]

X_val = validation_df[features]
y_val = validation_df[target]

X_test = test_df[features]
y_test = test_df[target]

In [11]:
categorical_features = [
    "ATM Name",
    "Festival Religion",
    "Working Day",
     "Holiday Sequence"
]

numerical_features = [
    "Lag_1_Day",
    "Lag_7_Day",
    "Lag_14_Day",
    "Lag_30_Day",
    "Rolling_7_Day_Mean",
    "Rolling_14_Day_Mean",
    "Rolling_30_Day_Mean",
    "Year",
    "Month",
    "Day",
    "Day_of_Week",
    "Is_Weekend",
    "Week_of_Year",
   
]

In [12]:
print(df["Holiday Sequence"].unique())
print(df["Holiday Sequence"].dtype)

['WHH' 'HHH' 'HHW' 'HWW' 'WWW' 'WWH' 'WHW' 'HWH']
object


#preprocessing pipeline

In [13]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numerical_features
        )
    ]
)

#Baseline=> Tomorrow's demand ≈ previous day's demand.

In [14]:
baseline_predictions = X_test["Lag_1_Day"]

baseline_mae = mean_absolute_error(
    y_test,
    baseline_predictions
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        baseline_predictions
    )
)

baseline_mape = mean_absolute_percentage_error(
    y_test,
    baseline_predictions
) * 100

baseline_r2 = r2_score(
    y_test,
    baseline_predictions
)

print("Baseline MAE:", baseline_mae)
print("Baseline RMSE:", baseline_rmse)
print("Baseline MAPE:", baseline_mape)
print("Baseline R²:", baseline_r2)

Baseline MAE: 3266.05550343965
Baseline RMSE: 4592.1801250281305
Baseline MAPE: 290.19935785849344
Baseline R²: 0.0811836564150561


In [16]:
print("Missing values in X_train:")
print(X_train.isnull().sum())

print("\nMissing values in X_test:")
print(X_test.isnull().sum())

Missing values in X_train:
ATM Name                 0
Lag_1_Day                0
Lag_7_Day                0
Lag_14_Day               0
Lag_30_Day             162
Rolling_7_Day_Mean       0
Rolling_14_Day_Mean      0
Rolling_30_Day_Mean      0
Year                     0
Month                    0
Day                      0
Day_of_Week              0
Is_Weekend               0
Week_of_Year             0
Working Day              0
Holiday Sequence         0
Festival Religion        0
dtype: int64

Missing values in X_test:
ATM Name                 0
Lag_1_Day                0
Lag_7_Day                0
Lag_14_Day               0
Lag_30_Day             203
Rolling_7_Day_Mean       0
Rolling_14_Day_Mean      0
Rolling_30_Day_Mean      0
Year                     0
Month                    0
Day                      0
Day_of_Week              0
Is_Weekend               0
Week_of_Year             0
Working Day              0
Holiday Sequence         0
Festival Religion        0
dtype: int64


In [17]:
print("\nRows containing NaN in X_train:")
print(X_train[X_train.isnull().any(axis=1)].head())


Rows containing NaN in X_train:
             ATM Name  Lag_1_Day  Lag_7_Day  Lag_14_Day  Lag_30_Day  \
0         Airport ATM    9811.90   11104.46    10772.76         NaN   
1  Christ College ATM    7982.20    7843.10     6156.78         NaN   
2      Mount Road ATM    9659.96   16963.78    16433.06         NaN   
3      Big Street ATM    1949.54    2752.04     2649.32         NaN   
4        KK Nagar ATM   18754.96   16736.94    20229.42         NaN   

   Rolling_7_Day_Mean  Rolling_14_Day_Mean  Rolling_30_Day_Mean  Year  Month  \
0         9288.517143          9561.367143          9561.367143  2011      1   
1         8572.228571          8889.407143          8889.407143  2011      1   
2        13323.028571         14585.934286         14585.934286  2011      1   
3         1939.145714          1927.528571          1927.528571  2011      1   
4        16585.305714         18534.998571         18534.998571  2011      1   

   Day  Day_of_Week  Is_Weekend  Week_of_Year Working Day H

#Linear regression

In [15]:
#Create
linear_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)
#Train
linear_model.fit(
    X_train,
    y_train
)
#Predict
linear_predictions = linear_model.predict(X_test)
#Evaluate
linear_mae = mean_absolute_error(
    y_test,
    linear_predictions
)
linear_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        linear_predictions
    )
)
linear_mape = mean_absolute_percentage_error(
    y_test,
    linear_predictions
) * 100
linear_r2 = r2_score(
    y_test,
    linear_predictions
)

print("Linear Regression")
print("MAE:", linear_mae)
print("RMSE:", linear_rmse)
print("MAPE:", linear_mape)
print("R²:", linear_r2)

ValueError: Input X contains NaN.
LinearRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [ ]:
for col in features:
    if df[col].dtype == "object":
        print("\n", col)
        print(df[col].unique()[:20])


 ATM Name
['Airport ATM' 'Christ College ATM' 'Mount Road ATM' 'Big Street ATM'
 'KK Nagar ATM']

 Working Day
['H' 'W']

 Holiday Sequence
['WHH' 'HHH' 'HHW' 'HWW' 'WWW' 'WWH' 'WHW' 'HWH']

 Festival Religion
['H' 'NH' 'N' 'M' 'C']
